In [1]:
import os
import json
import numpy as np
import time
import cv2
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm
import base64
from langchain_openai import ChatOpenAI
from langchain.schema.messages import HumanMessage, AIMessage
# from key import OPENAI_KEY   # Import your own OpenAI key.
from key import DASHSCOPE_API_KEY   # Import your own DashScope API Key.   阿里灵积（DashScope）接口
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [2]:
# text_dir = "../data/text/Subtask_2_test.json" # path to json file
# anno = json.load(open(text_dir))

text_dir = "../data/text/Subtask_2_train.json" # path to json file
anno = json.load(open(text_dir))
anno = anno[:100]
print("anno:", anno)

anno: [{'conversation_ID': 1, 'conversation': [{'utterance_ID': 1, 'text': 'Alright , so I am back in high school , I am standing in the middle of the cafeteria , and I realize I am totally naked .', 'speaker': 'Chandler', 'emotion': 'neutral', 'video_name': 'dia1utt1.mp4'}, {'utterance_ID': 2, 'text': 'Oh , yeah . Had that dream .', 'speaker': 'All', 'emotion': 'neutral', 'video_name': 'dia1utt2.mp4'}, {'utterance_ID': 3, 'text': 'Then I look down , and I realize there is a phone ... there .', 'speaker': 'Chandler', 'emotion': 'surprise', 'video_name': 'dia1utt3.mp4'}, {'utterance_ID': 4, 'text': 'Instead of ... ?', 'speaker': 'Joey', 'emotion': 'surprise', 'video_name': 'dia1utt4.mp4'}, {'utterance_ID': 5, 'text': 'That is right .', 'speaker': 'Chandler', 'emotion': 'anger', 'video_name': 'dia1utt5.mp4'}, {'utterance_ID': 6, 'text': 'Never had that dream .', 'speaker': 'Joey', 'emotion': 'neutral', 'video_name': 'dia1utt6.mp4'}, {'utterance_ID': 7, 'text': 'No .', 'speaker': 'Phoebe'

#### Creating images corresponding to each video utterances

In [15]:
def write_text(image, text, fname):
    """
    Function that writes given text on image and
    saves the file with name as fname.
    """
    width, height = 512, 192
    im = Image.open(image)
    draw = ImageDraw.Draw(im)
    
    text_width = width * 0.90
    text_max_height = height * 0.8
    
    size = 36
    # Dynamic resizing
    while size > 1:
        font_path = "open-sans/OpenSans-Regular.ttf" # Insert your own font path here
        font = ImageFont.truetype(font_path, size)
        lines = []
        line = ""
        for word in text.split():
          proposed_line = line
          if line:
            proposed_line += " "
          proposed_line += word
          if font.getlength(proposed_line) <= text_width:
            line = proposed_line
          else:
            # If this word was added, the line would be too long
            # Start a new line instead
            lines.append(line)
            line = word
        if line:
          lines.append(line)
        text = "\n".join(lines)
        
        x1, y1, x2, y2 = draw.multiline_textbbox((0, 0), text, font)
        w, h = x2 - x1, y2 - y1
        if h <= text_max_height:
          break
        else:
          # The text did not fit comfortably into the image
          # Try again at a smaller font size
          size -= 1
    
    draw.multiline_text((width / 2 - w / 2 - x1, height / 2 - h / 2 - y1), text, font=font, align="center", fill =(0, 0, 0))
    # im.save("captions/"+fname+".jpg
    im.save("captions1/"+fname+".jpg")

def make_video_grid(video_path, video_dir):
    """
    Reads the frames of video file at video_path and extracts 9 equidistant
    frames from video. It then organizes the 9 frames into a 3x3 grid to make up
    a single image. The text bar is added below the frame grid to make up the final
    image representing the video of the utterance along with its text.
    """
    # Open video
    video_id = video_path.split(".")[0]
    cap = cv2.VideoCapture(os.path.join(video_dir,video_path))
    # print("os.path.join(video_dir, video_path):", os.path.join(video_dir,video_path))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    # Grid size
    grid_size = 3
    
    # Frame skipping interval
    if frame_count >= 9:
       frame_skip = frame_count // 9
    else:
       print("frame count:", frame_count)
       frame_skip = 1
        
    # Initialize frame counter
    frame_num = 1
    
    # Initialize output image
    grid_image = np.zeros((frame_height*grid_size, frame_width*grid_size, 3), np.uint8)
    
    frame_buffer = []
    while cap.isOpened():
      ret, frame = cap.read()
    
      if ret == True:
          # Skip frames
          if frame_num % frame_skip == 0: frame_buffer.append(frame)
          frame_num += 1
          if len(frame_buffer) == grid_size*grid_size:
              break
      else:
          break
    cap.release()

    # Load text bar
    # text = cv2.imread("captions/"+video_id+".jpg")
    text = cv2.imread("captions1/"+video_id+".jpg")
    num_frames = 0
    for i in range(grid_size):
         for j in range(grid_size):
              if num_frames < len(frame_buffer):
                grid_image[i*frame_height:(i+1)*frame_height, j*frame_width:(j+1)*frame_width] = frame_buffer[i*grid_size+j]
                num_frames += 1
              else:
                break
      
    image = cv2.resize(grid_image, (512, 320))
    image = np.vstack([image, text])
    # cv2.imwrite("frames/"+video_id+".jpg", image)
    cv2.imwrite("frames1/"+video_id+".jpg", image)

In [16]:
# Creating a white bar which will be added below the frames
# Containing the speaker and utterance content.

bar = np.ones((512 - 320, 512, 3), np.uint8) * 255
cv2.imwrite("bar.jpg", bar)

# os.makedirs("frames", exist_ok=True)
# os.makedirs("captions", exist_ok=True)
os.makedirs("frames1", exist_ok=True)
os.makedirs("captions1", exist_ok=True)

# Creating all possible white bar to be added to each utterance images.
for an in anno:
  conv_id = an['conversation_ID']
  conversation = an['conversation']
  for utt in conversation:
    write_text("bar.jpg", f"{utt['speaker']}: \"{utt['text']}\"", f"dia{conv_id}utt{utt['utterance_ID']}")

# Creating images for all utterance videos
for an in tqdm(anno):
  conv_id = an['conversation_ID']
  conversation = an['conversation']
  for utt in conversation:
    video_path = utt['video_name']
    # if not os.path.exists("frames/" + video_path.split(".")[0] + ".jpg"):
    if not os.path.exists("frames1/" + video_path.split(".")[0] + ".jpg"):
      make_video_grid(video_path, "../datasets--dim--SemEvalSubtask2/training_data/all_videos_and_audios/train/")

 55%|█████████████████████████████████████████████                                     | 55/100 [01:07<01:03,  1.41s/it]

frame count: 8
frame count: 5
frame count: 3


 69%|████████████████████████████████████████████████████████▌                         | 69/100 [01:29<00:43,  1.41s/it]

frame count: 3
frame count: 7


 80%|█████████████████████████████████████████████████████████████████▌                | 80/100 [01:45<00:41,  2.07s/it]

frame count: 8


 83%|████████████████████████████████████████████████████████████████████              | 83/100 [01:47<00:18,  1.10s/it]

frame count: 7


100%|█████████████████████████████████████████████████████████████████████████████████| 100/100 [02:05<00:00,  1.25s/it]


In [ ]:
## 生成captions和frames文件夹 《--对应--》 captions1和frames1文件夹 对应 字幕和视频帧文件夹

#### Prompting GPT-4 Vision API

In [24]:
# chain = ChatOpenAI(model="gpt-4-vision-preview", openai_api_key=OPENAI_KEY)

# def get_caption(batch):
#     """
#     Prompts GPT-4 Vision API to describe a series of utterance images from a conversation where each utterance
#     image represents 9 frames in a 3x3 grid from the utterance video along with text below for what the speaker 
#     was saying.
#     """
#     out = chain.invoke(
#         [   
#             AIMessage(
#                 content="You are an expert of Friends TV Show. You can understand a video scene from a few of its frames shown in sequence. You give precise descriptive analysis."
#                 ),
#             HumanMessage(
#                     content=[
#                         "Describe what is likely going on in following images of video frames of each utterance in conversation. The caption below provides speaker context. Give output as:\n\
#                         Scene Description: {}",
#                         *map(lambda x: {"image": x}, batch),
#                     ]
#                 )
#         ]
#     )
#     return out.content

#### Prompting qwen-vl-plus API

#### Prompting qwen3.8-27b API

In [9]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

In [10]:
def get_conv_imgs_batched(conversation, batch_size=10):
    """
    Utterance images belonging to a conversation are batched to a particular batch_size.
    """
    conv_imgs = []
    for utt in conversation:
        # img_path = "frames/" + utt['video_name'].split(".")[0] + ".jpg"
        img_path = "frames1/" + utt['video_name'].split(".")[0] + ".jpg"
        conv_imgs.append(encode_image(img_path))
    
    k = int(np.ceil(len(conv_imgs)/float(batch_size)))
    batches = []
    for i in range(k):
        batches.append(conv_imgs[i*batch_size: (i+1)*batch_size])
    return batches

In [11]:
chain = ChatOpenAI(
                   # model="qwen-vl-plus", 
                   model="qwen3.8-27b",
                   openai_api_key=DASHSCOPE_API_KEY,
                   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")


def get_caption(batch):
    """
    Prompts qwen-vl-plus/qwen3.8-27b API to describe a series of utterance images from a conversation where each utterance
    image represents 9 frames in a 3x3 grid from the utterance video along with text below for what the speaker 
    was saying.
    """
    
    contents = []
   # 第一段是文本
    contents.append({
        "type": "text",
        "text": "Describe what is likely going on in following images of video frames of each utterance in conversation. The caption below provides speaker context. Give output as:\n\
                        Scene Description: {}"
    })
     # 依次加入每张图像
    for img_b64 in batch:
        contents.append({
            "type": "image_url",
            "image_url": f"data:image/jpeg;base64,{img_b64}"
        })
    try:    
        out = chain.invoke(
           input = [   
                AIMessage(
                    content="You are an expert of Friends TV Show. You can understand a video scene from a few of its frames shown in sequence. You give precise descriptive analysis."
                    ),
                HumanMessage(
                       content=contents
                    )
            ],
           config = {
            "extra_headers": {"X-DashScope-SafeMode": "NONE"}
                }
        )
        return out.content
    except Exception as e:
        print("在调用大语言模型时有异常发生。")
        print("异常类型为:", type(e))
        print("异常内容为:", e)
        return "Exceptions occur when calling large language models."


In [12]:
video_raw_out = {}
# video_raw_out = json.load(open("eval_raw_out.json")) # if resuming

调用 GPT-4 Vision 生成视频描述

In [61]:
# Processing all the conversations to get video captions
# Save steps and sleeps are to make sure that progress is not lost due to hitting
# GPT-4 API Rate Limits. Please be careful!!

# save_step = 10
# for i,an in enumerate(anno):
#     conv_id = an['conversation_ID']
#     if str(conv_id) in video_raw_out or conv_id in video_raw_out: continue
#     batches = get_conv_imgs_batched(an['conversation'])
#     outputs = []
#     for batch in tqdm(batches):
#         out = get_caption(batch)
#         outputs.append(out)
#         time.sleep(0.5)
#     video_raw_out[conv_id] = outputs
#     print("[{}/{}] Processed Conv {}".format(i+1, len(anno), conv_id))
#     if i%save_step == 0:
#         print("json dump...")
#         json.dump(video_raw_out, open("eval_raw_out.json", "w"))

In [13]:
# Processing all the conversations to get video captions
# Save steps and sleeps are to make sure that progress is not lost due to hitting
# qwen-vl-plus/qwen3.8-27b API Rate Limits. Please be careful!!

save_step = 10
for i,an in enumerate(anno):
    conv_id = an['conversation_ID']
    if str(conv_id) in video_raw_out or conv_id in video_raw_out: continue
    batches = get_conv_imgs_batched(an['conversation'])
    outputs = []
    for batch in tqdm(batches):
        out = get_caption(batch)
        if out is None:
            # 遇到屏蔽内容 → 在输出中标记为空字符串，保证结构完整
            outputs.append("SKIPPED_DUE_TO_CONTENT_RESTRICTION")
        else:
            outputs.append(out)
        time.sleep(0.5)
    video_raw_out[conv_id] = outputs
    print("[{}/{}] Processed Conv {}".format(i+1, len(anno), conv_id))
    if i%save_step == 0:
        print("json dump...")
        # json.dump(video_raw_out, open("eval_raw_out.json", "w"))
        json.dump(video_raw_out, open("eval_raw_out1.json", "w"))

100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:25<00:00, 25.77s/it]


[1/100] Processed Conv 1
json dump...


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:30<00:00, 30.47s/it]


[2/100] Processed Conv 2


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:39<00:00, 39.31s/it]


[3/100] Processed Conv 3


... (192 outputs left out)

100%|█████████████████████████████████████████████████████████████████████████████████████| 2/2 [01:13<00:00, 36.71s/it]


[99/100] Processed Conv 128


100%|█████████████████████████████████████████████████████████████████████████████████████| 1/1 [01:15<00:00, 75.15s/it]

[100/100] Processed Conv 129


In [14]:
# json.dump(video_raw_out, open("eval_raw_out.json", "w"))
json.dump(video_raw_out, open("eval_raw_out1.json", "w"))

#### Postprocessing Raw Batched Video Captions to Coherent Caption for Whole Conversation

In [3]:
# video_raw_out = json.load(open("eval_raw_out.json"))
video_raw_out = json.load(open("eval_raw_out1.json"))

In [4]:
conv_ids = []
descs = []
for conv_id, outs in video_raw_out.items():
    desc = "\n".join(outs)
    conv_ids.append(conv_id)
    descs.append({"description": desc})

In [3]:
summary_prompt = \
"""
Following is a descriptions of video clip from Friends TV show for a particular conversation.
The descriptions are broken from each other. Stitch the description into a continous coherent
narrative of the whole scene.
{description}
"""
summary_prompt = ChatPromptTemplate.from_template(summary_prompt)

# model = ChatOpenAI(openai_api_key=OPENAI_KEY)
model = ChatOpenAI(
                   # model="qwen-plus-latest",
                   # model="qwen3.8-2.4t-a95b", 
                   # model="qwen3.8-max", 
                   # model="qwen3.8-max-0902",
                   model="qwen3.8-flash",
                   openai_api_key=DASHSCOPE_API_KEY,
                   base_url="https://dashscope.aliyuncs.com/compatible-mode/v1")
output_parser = StrOutputParser()
description_chain = summary_prompt | model | output_parser

In [ ]:
# outs = description_chain.batch(descs, config={"max_concurrency": 10})

# 限制并发到 1 或 2（最安全）
outs = description_chain.batch(
    descs,
    config={"max_concurrency": 1}   # 或 2
)

In [ ]:
video_proc_out = {}
for i, out in enumerate(outs):
    video_proc_out[conv_ids[i]] = out    

In [ ]:
# json.dump(video_proc_out, open("eval_proc_out.json", "w"))
json.dump(video_proc_out, open("eval_proc_out1.json", "w"))

In [1]:
!pip install tqdm

In [4]:
#### 每次只处理一条
import json
import os
import time
from tqdm import tqdm

# 输入文件
input_file = "eval_raw_out1.json"

# 输出文件
output_file = "eval_proc_out1.json"

# 加载视频描述
with open(input_file, "r", encoding="utf-8") as f:
    video_raw_out = json.load(f)

conv_ids = []
descs = []

for conv_id, outs in video_raw_out.items():
    desc = "\n".join(outs)
    conv_ids.append(conv_id)
    descs.append({"description": desc})

# 加载已经完成的任务，实现断点续跑
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        video_proc_out = json.load(f)
else:
    video_proc_out = {}

# 过滤已经完成的任务
tasks = [
    (conv_id, desc)
    for conv_id, desc in zip(conv_ids, descs)
    if conv_id not in video_proc_out
]

print(f"总任务数量：{len(conv_ids)}")
print(f"已经完成：{len(video_proc_out)}")
print(f"待处理任务：{len(tasks)}")

# 每处理多少条保存一次
# SAVE_INTERVAL = 5
SAVE_INTERVAL = 1

# 统计失败任务
failed_tasks = []

# 逐条处理并显示进度
with tqdm(
    tasks,
    desc="Processing videos",
    unit="video",
    dynamic_ncols=True
) as pbar:

    for i, (conv_id, desc) in enumerate(pbar, 1):

        try:
            # 调用模型
            result = description_chain.invoke(desc)

            # 保存结果
            video_proc_out[conv_id] = result

            pbar.set_postfix({
                "success": len(video_proc_out),
                "failed": len(failed_tasks)
            })

        except Exception as e:
            print(f"\n处理失败：{conv_id}")
            print(f"错误信息：{e}")

            failed_tasks.append(conv_id)

        # 定期保存
        if i % SAVE_INTERVAL == 0:
            with open(output_file, "w", encoding="utf-8") as f:
                json.dump(
                    video_proc_out,
                    f,
                    ensure_ascii=False,
                    indent=2
                )

# 全部任务结束后保存
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(
        video_proc_out,
        f,
        ensure_ascii=False,
        indent=2
    )

print("\n========== 任务处理完成 ==========")
print(f"总任务数量：{len(conv_ids)}")
print(f"成功数量：{len(video_proc_out)}")
print(f"失败数量：{len(failed_tasks)}")
print(f"结果文件：{output_file}")

if failed_tasks:
    print("失败的 conversation ID：")
    print(failed_tasks)


总任务数量：100
已经完成：58
待处理任务：42


Processing videos: 100%|██████████████████████████████████████| 42/42 [32:04<00:00, 45.81s/video, success=100, failed=0]


========== 任务处理完成 ==========
总任务数量：100
成功数量：100
失败数量：0
结果文件：eval_proc_out1.json


In [4]:
### 每次只处理十条
from tqdm import tqdm

# 输入文件
input_file = "eval_raw_out1.json"

# 输出文件
output_file = "eval_proc_out2.json"

# 加载视频描述
with open(input_file, "r", encoding="utf-8") as f:
    video_raw_out = json.load(f)

conv_ids = []
descs = []

for conv_id, outs in video_raw_out.items():
    desc = "\n".join(outs)
    conv_ids.append(conv_id)
    descs.append({"description": desc})

# 加载已经完成的任务，实现断点续跑
if os.path.exists(output_file):
    with open(output_file, "r", encoding="utf-8") as f:
        video_proc_out = json.load(f)
else:
    video_proc_out = {}
    
batch_size = 10
outs = []

for start in tqdm(
    range(0, len(descs), batch_size),
    desc="Processing batches",
    unit="batch"
):
    batch_descs = descs[start:start + batch_size]

    batch_outs = description_chain.batch(
        batch_descs,
        config={"max_concurrency": 1}
    )

    outs.extend(batch_outs)

    # 实时保存中间结果
    for j, out in enumerate(batch_outs):
        video_proc_out[conv_ids[start + j]] = out

    with open("eval_proc_out2.json", "w", encoding="utf-8") as f:
        json.dump(
            video_proc_out,
            f,
            ensure_ascii=False,
            indent=2
        )
### 不支持断点续跑噢 ~

Processing batches: 100%|█████████████████████████████████████████████████████████| 10/10 [1:09:33<00:00, 417.36s/batch]
